# <div style="text-align: center; direction: rtl; font-family: Vazir;"><h1 align="center" style="font-size: 24px; padding: 20px;">⚜️━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━⚜️<br>بهبود توانایی مدل های زبانی بزرگ در پیروی از دستورالعمل<br>⚜️━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━⚜️</h1></div>


## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش اول: آشنایی با داده و مدل (15 نمره)</div>


### <div style="direction: rtl; font-family: Vazir;">بارگذاری و بررسی مجموعه‌داده</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">

در این بخش، هدف آشنایی اولیه با داده‌های آموزشی مبتنی بر دستورالعمل است.<br>
از دیتاست زیر استفاده می‌کنیم:

- `allenai/tulu-3-sft-personas-instruction-following`

ابتدا دیتاست را از **[Huggingface](https://huggingface.co/)** بارگذاری کرده و یک نمونه را نمایش دهید. سپس بصورت مختصر، هدف دیتاست، نوع داده‌ها و ساختار مکالمه‌ها را بیان کنید.
(نیازی به ورود به جزئیات نیست)

✍️ **پاسخ:**

هدف این دیتاست این است که مدل‌های زبانی را در **instruction-following دقیق** تقویت کند؛ یعنی مدل بتواند طبق یک دستور متنی، با رعایت چند قید مشخص (مثلاً فرمت، تعداد کلمه، وجود یا عدم وجود بعضی کلمات و …) جواب تولید کند.

نوع داده‌ها به‌صورت **نمونه‌های مکالمه‌ای (chat-style)** است. هر نمونه شامل یک شناسه (`id`)، یک دستور یا پرامپت اصلی (`prompt`) و یک لیست پیام‌ها (`messages`) است که دیالوگ بین «کاربر» و «دستیار» را نشان می‌دهد؛ معمولاً یک پیام user و یک پاسخ assistant که برای SFT استفاده می‌شود. علاوه بر این، یک لیست `constraints` هم وجود دارد که در آن قیدهای قابل‌ارزیابی برای همان دستور نوشته شده است.

به‌طور خلاصه، ساختار مکالمه‌ها شبیه گفت‌وگوی کاربر–دستیار است: کاربر یک دستور نسبتاً پیچیده با ۱ تا ۳ قید می‌دهد و دستیار باید پاسخی تولید کند که هم از نظر محتوا درست باشد و هم تمام این قیود را تا حد ممکن رعایت کند. این فرمت مستقیماً برای Supervised Fine-Tuning مدل‌های چت (LLMهای instruction-tuned) طراحی شده است.

</div>

In [1]:
dataset_name = "allenai/tulu-3-sft-personas-instruction-following"
sample_size = 2000

from datasets import load_dataset

raw_dataset = load_dataset(dataset_name)
train_ds = raw_dataset["train"]
train_ds = train_ds.shuffle(seed=42).select(range(sample_size))

print(train_ds)
print("Columns:", train_ds.column_names)

example = train_ds[0]
print("\nExample keys:", example.keys())
print("\nID:", example.get("id", None))
print("\nPrompt:\n", example.get("prompt", None))
print("\nConstraints:\n", example.get("constraints", None))
print("\nMessages:")
for m in example["messages"]:
    print(f"- role: {m['role']}\n  content: {m['content'][:200]}...\n")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/39.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29980 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'messages', 'constraints'],
    num_rows: 2000
})
Columns: ['id', 'prompt', 'messages', 'constraints']

Example keys: dict_keys(['id', 'prompt', 'messages', 'constraints'])

ID: personas_IF_9326uy2wpt2sz9k5rxumrsfy

Prompt:
 Please write an analysis of the romantic dynamics between the main characters in a popular rom-com of your choice. Make sure to discuss how the characters' backgrounds and cultural differences contribute to their relationship development. Include a postscript where you suggest another rom-com movie that offers a contrasting romantic dynamic.

Constraints:
 ['content:include a postscript']

Messages:
- role: user
  content: Please write an analysis of the romantic dynamics between the main characters in a popular rom-com of your choice. Make sure to discuss how the characters' backgrounds and cultural differences contrib...

- role: assistant
  content: **Analysis of Romantic Dynamics in "Crazy Rich Asians"**

"Crazy Rich Asi

### <div style="direction: rtl; font-family: Vazir;">مقایسه توکنایزرهای مختلف</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">

قبل از شروع کار با مدل، مهم است که نحوه کار توکنایزر (Tokenizer) را درک کنیم. توکنایزر ها نقش حیاتی در عملکرد مدل دارند و توکنایزر های مختلف می‌توانند متن یکسان را به روش‌های متفاوتی تجزیه کنند.

مراحل زیر را انجام دهید:

1. ایتدا یک توکن دسترسی Huggingface از آدرس زیر ایجاد کنید:
https://huggingface.co/settings/tokens

1. توکنایزر را برای مدل‌های زیر بارگذاری کنید:
    - `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
    - `google/gemma-3-4b-it`
    - `meta-llama/Llama-3.1-8B`

2. متن `sample_text1` را بازای هر سه توکنایزر پردازش کرده و نمایش دهید.

3. یک جمله فارسی با **حداقل ۱۰ کلمه** در نظر گرفته و این کار را تکرار کنید.

هنگام استخراج توکن‌ها، به‌گونه‌ای عمل کنید که نشانه‌های
   ویژه‌ حفظ شود.

</div>

In [2]:
from IPython.display import display, HTML
from transformers import AutoTokenizer

TOKEN_BG_COLORS = [
    "#58a6ff", "#f78166", "#85e89d", "#ffab70",
    "#bca0ff", "#79c0ff", "#ffb4a2", "#d0d0d0",
]

def show_raw_tokens(model_name, token_list):
    html_parts = [
        '<div style="margin:20px 0; padding:15px; background:#1e1e1e; border-radius:10px; '
        'border-left:6px solid #6272a4; font-family: monospace;">'
        f'<strong style="color:#ff79c6; font-size:18px;">{model_name}</strong><br><br>'
        '<div style="line-height: 1.6; white-space: pre-wrap;">'
    ]

    for i, token in enumerate(token_list):
        color = TOKEN_BG_COLORS[i % len(TOKEN_BG_COLORS)]
        style = (
            f"background:{color}; color:black; padding:2px 2px; margin:2px; "
            f"border-radius:2px; display:inline-block; font-size:13px;"
        )

        # Only escape < and > to prevent HTML breaking
        safe_token = token.replace("<", "&lt;").replace(">", "&gt;")

        html_parts.append(f'<span style="{style}">{safe_token}</span>')

    html_parts.append('</div></div>')
    display(HTML("".join(html_parts)))

In [3]:
sample_text1 = """English and CAPITALIZATION 🎵 漢字
show_tokens False None elif == >= else:
two tabs:"	" Three tabs: "			"
3.25 * 12 = 39"""

sample_text2 = (
    "امروز صبح همراه سه تا از دوستانم در دانشگاه درباره یادگیری ماشین "
    "و شبکه‌های عصبی عمیق صحبت کردیم."
)


from transformers import AutoTokenizer

hf_token = "hf_XXX" #Masked Token :)
model_names = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
]

tokenizers = {}
for name in model_names:
    print(f"Loading tokenizer for: {name}")
    tokenizer = AutoTokenizer.from_pretrained(
        name,
        token=hf_token,
    )
    tokenizers[name] = tokenizer

def tokenize_and_show(text, description):
    print("\n" + "=" * 80)
    print(f"{description}")
    print("-" * 80)
    print(text)
    print("=" * 80 + "\n")

    for model_name, tokenizer in tokenizers.items():
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            return_tensors=None,
        )

        input_ids = encoded["input_ids"]
        if isinstance(input_ids[0], list):
            input_ids = input_ids[0]

        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        show_raw_tokens(model_name, tokens)

tokenize_and_show(sample_text1, "Tokenization for sample_text1")

tokenize_and_show(sample_text2, "Tokenization for sample_text2 (Persian)")


Loading tokenizer for: TinyLlama/TinyLlama-1.1B-Chat-v1.0


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]


Tokenization for sample_text1
--------------------------------------------------------------------------------
English and CAPITALIZATION 🎵 漢字
show_tokens False None elif == >= else:
two tabs:"	" Three tabs: "			"
3.25 * 12 = 39




Tokenization for sample_text2 (Persian)
--------------------------------------------------------------------------------
امروز صبح همراه سه تا از دوستانم در دانشگاه درباره یادگیری ماشین و شبکه‌های عصبی عمیق صحبت کردیم.



<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>

⭐ <b>سؤال:</b>

با توجه به خروجی‌های به‌دست‌آمده، به‌نظر شما کدام‌یک از این سه مدل در پردازش زبان فارسی عملکرد بهتری دارد؟<br>
چرا این مدل‌ها یک متن یکسان را به‌صورت متفاوت توکنایز می‌کنند؟

✍️ <b>پاسخ تشریحی:</b>

با توجه به تصویر، روی جمله‌ی فارسی می‌بینیم که در TinyLlama تقریباً هر حرفِ فارسی یک توکن جداست (کادرهای ریز پشت‌سرهم)، در حالی‌که در Gemma هر توکن معمولاً یک کلمه یا زیرکلمه‌ی قابل‌فهم مثل «▁ام»، «روز»، «▁دوستانم»، «▁دانشگاه»، «▁ماشین‌گیری» است. در Llama-3.1-8B هم توکن‌ها به صورت رشته‌های عجیب مثل `Ø§Ù…` دیده می‌شوند که در واقع تکه‌های بایت هستند و اصلاً شبیه کلمات/حروف فارسی نیستند. بنابراین برای پردازش فارسی، از بین این سه مدل <b>Gemma بهترین رفتار را دارد</b>.

این‌که سه مدل یک متن یکسان را متفاوت توکنایز می‌کنند به این دلیل است که هرکدام:
۱) واژگان (vocab) مخصوص خودشان را دارند که از روی داده‌ی پیش‌تمرین ساخته شده؛
۲) از الگوریتم توکن‌سازی متفاوتی استفاده می‌کنند (مثلاً Gemma شبیه SentencePiece روی کلمه/زیرکلمه، Llama-3.1 روی بایت‌های UTF-8)،
۳) و میزان پوشش زبان فارسی در داده‌ی آموزش آن‌ها فرق می‌کند. به همین دلیل یک جمله‌ی واحد در هر مدل با تعداد و نوع توکن‌های متفاوت نمایش داده می‌شود.

</div>

### <div style="direction: rtl; font-family: Vazir;">بارگذاری مدل</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">

در این تمرین از مدل `TinyLlama-1.1B-Chat-v1.0` استفاده می‌کنیم.

این مدل یک LLM سبک با حدود 1.1 میلیارد پارامتر است و برای کاربردهای مکالمه‌ای تنظیم شده است. مدل و tokenizer آن را بارگذاری کنید.

</div>


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
 )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
⭐ <b>سؤال:</b>
در صورتیکه torch_dtype را از نوع float16 یا float32 تعریف کنیم، میزان حافظه VRAM استفاده شده توسط مدل چه تفاوتی میکند؟

✍️ <b>پاسخ تشریحی:</b><br>
در float32 هر پارامتر مدل با ۴ بایت ذخیره می‌شود، اما در float16 هر پارامتر فقط ۲ بایت می‌گیرد. یعنی برای خودِ وزن‌های مدل، استفاده از <b>float16 تقریباً نصف float32 حافظه VRAM مصرف می‌کند</b>.
البته علاوه‌بر وزن‌ها، چیزهایی مثل activationها و cache هم روی VRAM اثر دارند، ولی به‌طور کلی اگر همین مدل را یک‌بار با float32 و یک‌بار با float16 لود کنیم، نسخه‌ی float16 حدوداً ۲ برابر سبک‌تر است و روی GPUهای با حافظه‌ی کمتر راحت‌تر جا می‌شود.

</div>

### <div style="direction: rtl; font-family: Vazir;">بررسی Chat Template</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
ساختار Chat template مدل TinyLlama را بررسی کنید.
</div>


In [5]:
print("Chat template string:")
print(tokenizer.chat_template)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain overfitting in simple terms."},
]

formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\nFormatted conversation:")
print(formatted)


Chat template string:
{% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}

Formatted conversation:
<|system|>
You are a helpful assistant.</s>
<|user|>
Explain overfitting in simple terms.</s>
<|assistant|>



<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
⭐ <b>سؤال:</b>
چرا رعایت کردن chat template برای مدل‌های instruction-tuned ضروری است؟

✍️ <b>پاسخ تشریحی:</b><br>

در مرحله‌ی instruction-tuning، مدل TinyLlama تمام داده‌های آموزشی‌اش را در قالب یک chat template مشخص دیده است؛ یعنی همیشه دیالوگ‌ها به‌صورت ثابت مثل:

`system → user → assistant`
به همراه توکن‌های ویژه و جداکننده‌های خاص، به مدل داده شده‌اند.

اگر در زمان inference همین قالب را رعایت نکنیم (مثلاً بدون نقش‌ها یا بدون توکن‌های خاص به مدل متن بدهیم)، ورودی ما برای مدل «خارج از توزیع» می‌شود؛ مدل دیگر به‌درستی تشخیص نمی‌دهد کدام قسمت دستور کاربر است، کجا باید جواب بدهد و چطور باید رفتار چتی داشته باشد.

به همین دلیل، رعایت کردن chat template برای مدل‌های instruction-tuned ضروری است؛ چون مدل دقیقاً روی همین شکل ورودی آموزش دیده و توانایی instruction-following و نقش‌فهمی‌اش به این قالب وابسته است.

</div>

## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش دوم: آماده‌سازی و پیکربندی (10 نمره)</div>

### <div style="direction: rtl; font-family: Vazir;">آماده سازی داده برای SFT</div>

In [6]:
MAX_SEQ_LENGTH = 1024

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess_for_sft(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    tokenized = tokenizer(
        text,
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding="max_length",
    )

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    labels = [
        tok if tok != tokenizer.pad_token_id else -100
        for tok in input_ids
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_sft = train_ds.map(
    preprocess_for_sft,
    remove_columns=train_ds.column_names,
)

train_sft[0]


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'input_ids': [1,
  529,
  29989,
  1792,
  29989,
  29958,
  13,
  12148,
  2436,
  385,
  7418,
  310,
  278,
  6017,
  7716,
  19753,
  1546,
  278,
  1667,
  4890,
  297,
  263,
  5972,
  6017,
  29899,
  510,
  310,
  596,
  7348,
  29889,
  8561,
  1854,
  304,
  5353,
  920,
  278,
  4890,
  29915,
  3239,
  29879,
  322,
  16375,
  12651,
  29126,
  304,
  1009,
  9443,
  5849,
  29889,
  512,
  2325,
  263,
  1400,
  2154,
  988,
  366,
  4368,
  1790,
  6017,
  29899,
  510,
  14064,
  393,
  16688,
  263,
  12814,
  292,
  6017,
  7716,
  7343,
  29889,
  2,
  29871,
  13,
  29966,
  29989,
  465,
  22137,
  29989,
  29958,
  13,
  1068,
  21067,
  4848,
  310,
  6033,
  7716,
  22554,
  1199,
  297,
  376,
  29907,
  336,
  1537,
  4385,
  319,
  1039,
  550,
  29908,
  1068,
  13,
  13,
  29908,
  29907,
  336,
  1537,
  4385,
  319,
  1039,
  550,
  1699,
  10624,
  491,
  9937,
  341,
  29889,
  678,
  29884,
  29892,
  338,
  263,
  6017,
  29899,
  510,
  393,
  3902,


### <div style="direction: rtl; font-family: Vazir;">تعریف پیکربندی LoRA</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
<p style="line-height: 1.8; text-align: right;">
روش های مختلفی برای فاین تیون کردن مدل های شبکه عصبی وجود دارد. LoRA یک روش فاین‌تیون‌ سبک و کم‌هزینه است که با افزودن لایه‌های کوچک آموزش‌پذیر به برخی ماژول‌های مدل، امکان تنظیم دقیق LLMها را بدون آموزش کامل تمام وزن‌ها فراهم می‌کند.

در این بخش لازم است پیکربندی LoRA را برای مدل پایه مشخص کنید و سپس مدل پایه را با استفاده از آن به یک مدل قابل‌فاین‌تیون تبدیل کنید. در پایان، تعداد پارامترهای آموزش‌پذیر را گزارش کنید.
</p>
</div>


In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

def print_trainable_parameters(model):
    trainable_params = 0
    total_params = 0
    for p in model.parameters():
        num = p.numel()
        total_params += num
        if p.requires_grad:
            trainable_params += num
    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")

print_trainable_parameters(model)


Trainable params: 2,252,800
Total params: 1,102,301,184
Trainable %: 0.2044%


<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
⭐ <b>سؤال:</b>
منطق شما برای انتخاب مقادیر rank، α و target_modules چه بوده است؟

✍️ <b>پاسخ تشریحی:</b><br>

در این تنظیمات، مقدار `r=8` را انتخاب کردیم تا یک rank نسبتاً کوچک ولی قابل‌بیان داشته باشیم؛ یعنی آداپتورهای LoRA ظرفیت کافی برای یادگیری الگوهای جدید داشته باشند، اما تعداد پارامترهای آموزش‌پذیر خیلی زیاد نشود. همان‌طور که در گزارش دیده می‌شود، فقط حدود ۰٫۲٪ از کل پارامترهای مدل trainable هستند، که برای یک مدل ۱.۱B بسیار سبک و کم‌هزینه است.

مقدار `lora_alpha=16` را طوری انتخاب کردیم که نسبت `alpha / r ≈ 2` باشد. این نسبت در کارهای LoRA یک انتخاب رایج است و باعث می‌شود به‌روزرسانی‌های LoRA به‌اندازه‌ی کافی تقویت شوند، بدون این‌که ناپایداری شدید در آموزش ایجاد شود.

برای `target_modules`، ماژول‌های `["q_proj", "k_proj", "v_proj", "o_proj"]` را انتخاب کردیم، چون این لایه‌ها بخش اصلی مکانیزم attention در هر بلاک ترنسفورمر هستند و نحوه‌ی ترکیب اطلاعات از توکن‌های مختلف را کنترل می‌کنند. اعمال LoRA روی این projectionها باعث می‌شود مدل بتواند روش «خواندن و وزن‌دادن به context» را با داده‌ی جدید (instruction-following) تطبیق دهد، بدون این‌که لازم باشد همه‌ی وزن‌های شبکه را فاین‌تیون کنیم.

</div>

### <div style="direction: rtl; font-family: Vazir;">تنظیم پارامترهای آموزش</div>

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="tinyllama_lora_sft",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
)


## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش سوم: آموزش مدل (10 نمره)</div>


### <div style="direction: rtl; font-family: Vazir;">اجرای SFT با LoRA</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
در این بخش، مدل پایه را با استفاده از داده‌های آماده‌شده و پیکربندی LoRA آموزش می‌دهید.
هدف این مرحله اجرای Supervised Fine-Tuning و مشاهده تأثیر آن بر توانایی مدل در پیروی از دستورالعمل‌ها است.
</div>

In [9]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    data_collator=data_collator,
)

trainer.train()


The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,1.471900
20,1.359400
30,1.353800
40,1.327700
50,1.305400
60,1.284100
70,1.307000
80,1.292300
90,1.302600
100,1.271700


TrainOutput(global_step=125, training_loss=1.3235453300476074, metrics={'train_runtime': 874.1576, 'train_samples_per_second': 2.288, 'train_steps_per_second': 0.143, 'total_flos': 1.2739770580992e+16, 'train_loss': 1.3235453300476074, 'epoch': 1.0})

### <div style="direction: rtl; font-family: Vazir;">ذخیره‌سازی و ادغام وزن‌های LoRA</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
پس از پایان آموزش:

- وزن‌های آداپتور LoRA را ذخیره کنید.

- وزن‌های آداپتور را با مدل پایه ادغام کرده و نسخه کاملِ ادغام‌شده (merged model) را ذخیره کنید.

در پایان، اندازه فایل آداپتور و اندازه مدل ادغام‌شده را مقایسه کنید.
</div>


In [10]:
import os

adapter_dir = "tinyllama_lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

merged_dir = "tinyllama_lora_merged"
merged_model = model.merge_and_unload()
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

def get_dir_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

adapter_size_mb = get_dir_size(adapter_dir) / (1024 ** 2)
merged_size_mb = get_dir_size(merged_dir) / (1024 ** 2)

print(f"Adapter size (LoRA only): {adapter_size_mb:.2f} MB")
print(f"Merged model size: {merged_size_mb:.2f} MB")


Adapter size (LoRA only): 12.55 MB
Merged model size: 2102.13 MB


<p dir='rtl' style="line-height: 2.0; text-align: right; font-family: Vazir; font-size: 16px; margin-top: 20px; color: white; background-color:rgb(0, 40, 30); padding: 30px; border-radius: 8px;">
🎯 <b>خروجی مورد انتظار:</b><br>
شامل: اجرای موفق آموزش، فایل آداپتور، فایل مدل merged

</p>

## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش چهارم: ارزیابی عملکرد (15 نمره)</div>


### <div style="direction: rtl; font-family: Vazir;">ارزیابی کیفی</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
خروجی نسخه پایه و نسخه آموزش‌دیده مدل روی پرامپت‌های زیر را مقایسه نمایید.

✍️ <b>پاسخ:</b>

در پرامپت‌های ساده‌ مثل «یک اسلوگان برای اپ بلوز وینتیج بساز»، هر دو مدل (پایه و فاین‌تیون‌شده) اسلوگان‌های معنادار و مرتبط تولید می‌کنند، اما مدل فاین‌تیون‌شده معمولاً بعد از اسلوگان، توضیح توصیفی اضافه می‌کند؛ مثلاً شروع می‌کند به توضیح این‌که چرا این شعار خوب است و چه چیزی را منتقل می‌کند. این نشان می‌دهد که بعد از SFT، مدل به سمت سبک «جواب + توضیح» سوق داده شده، در حالی که مدل پایه اغلب یک جمله‌ی کوتاه‌تر و مستقیم‌تر می‌دهد.

در پرامپت دوم که فقط قید حروف بزرگ (ALL CAPS) را دارد، مدل پایه این قید را رعایت نمی‌کند و یک اسلوگان عادی با حروف معمولی می‌دهد، اما مدل فاین‌تیون‌شده تا حد خوبی به دستور عمل می‌کند و اسلوگان را کاملاً با حروف بزرگ تولید می‌کند؛ هرچند باز هم بر خلاف دستور پرامپت، توضیح اضافی بعد از اسلوگان می‌نویسد. این تفاوت نشان می‌دهد که فاین‌تیون کردن روی داده‌ی instruction-following حساسیت مدل به قیدهای صریح را بیشتر کرده، ولی عادت «توضیح دادن» را هم در آن تقویت کرده است.

در پرامپت‌های سخت‌تر (همه حروف بزرگ، پایان با واژه‌ی SOUL و دقیقاً هفت کلمه)، هر دو مدل در عمل این قیدها را به‌خوبی رعایت نمی‌کنند؛ نه مدل پایه اسلوگان را تماماً به حروف بزرگ و با کلمه‌ی soul تمام می‌کند، نه مدل فاین‌تیون‌شده. مدل فاین‌تیون‌شده معمولاً یک عبارت ثابت مثل «THE SOUL OF BLUES RECORDS» یا نسخه‌های مشابه را تکرار می‌کند که نه دقیقاً هفت کلمه است و نه با SOUL تمام می‌شود، اما از نظر سبک و حس کلی به محدودیت‌ها نزدیک‌تر است (مثلاً همه‌ی حروف بزرگ‌اند). در عوض مدل پایه تنوع بیشتری در اسلوگان‌ها دارد ولی به قیود دستور کمتر توجه می‌کند.

در آخرین پرامپت که علاوه بر قیدهای قبلی، فرمت JSON هم خواسته شده، مدل پایه اصلاً وارد JSON نمی‌شود و فقط یک اسلوگان معمولی می‌دهد؛ در حالی که مدل فاین‌تیون‌شده علاوه بر شعار، سعی می‌کند یک بلوک JSON برگرداند (مثلاً شیئی با کلیدهایی مثل `"slogan"` یا `"title"`)، اما باز هم طبق عادت، توضیح متنی اضافه هم بعد از JSON می‌نویسد. این رفتار نشان می‌دهد که فاین‌تیون، مدل را از نظر درک قالب‌های ساختاری (مثل JSON) و رعایت سبک خروجی منظم‌تر کرده، ولی هنوز در اجرای دقیق همه‌ی constraintها (تعداد کلمات، کلمه‌ی آخر، حذف کامل توضیح اضافه) کاملاً موفق نیست. در مجموع می‌توان گفت مدل فاین‌تیون‌شده نسبت به مدل پایه، از نظر درک و رعایت دستورالعمل‌ها یک گام جلوتر است، اما در مقابل، خروجی‌هایش طولانی‌تر، توضیحی‌تر و از نظر تنوع خلاقیت کمی محدودتر شده‌اند.

</div>

In [18]:
import torch
from transformers import AutoModelForCausalLM

prompts = [
    "Create a slogan for a new vintage blues record collection app.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters and end with the word soul.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters, end with the word soul, and contain exactly seven words.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters, end with the word soul, and contain exactly seven words. Wrap the entire answer in JSON format.",
]

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cpu",
)
base_model.eval()

ft_model = AutoModelForCausalLM.from_pretrained(
    "tinyllama_lora_merged",
    torch_dtype=torch.float16,
    device_map="auto",
)
ft_model.eval()

def generate_response(model, tokenizer, prompt, max_new_tokens=64):
    messages = [
        {
            "role": "user",
            "content": prompt + "\nOnly output the final slogan.Do not add explanations or extra text.",
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )

    generated_ids = outputs[0, inputs.shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return text.strip()

for p in prompts:
    print("=" * 80)
    print("PROMPT:")
    print(p)
    print("-" * 80)
    print("Base model:")
    print(generate_response(base_model, tokenizer, p))
    print("\nFine-tuned model:")
    print(generate_response(ft_model, tokenizer, p))
    print()


PROMPT:
Create a slogan for a new vintage blues record collection app.
--------------------------------------------------------------------------------
Base model:
"Discover the Blues: A Journey Through Time"

Fine-tuned model:
"Discover the Blues Like Never Before!"

This slogan perfectly captures the essence of the vintage blues record collection app, which aims to provide users with an immersive and personalized listening experience. The slogan emphasizes the app's focus on discovering and

PROMPT:
Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters.
--------------------------------------------------------------------------------
Base model:
"Discover the Blues - One Record at a Time"

Fine-tuned model:
"RELAX AND DISCOVER THE BEST OF BLUES HISTORY IN ONE APP."

This slogan is designed to capture the essence of the vintage blues record collection app, which aims to provide users with an immersive experience exploring the ri

### <div style="direction: rtl; font-family: Vazir;">ارزیابی کمی با معیار `ifeval`</div>

<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">

برای ارزیابی دقیق قابلیت instruction-following مدل، از بنچمارک **IFEval** استفاده می‌کنیم.

- مقاله: https://arxiv.org/abs/2311.07911
</div>

<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>

⭐ <b>سؤال:</b>
 بصورت مختصر بیان کنید که بنچمارک <b>ifeval</b> چه نوع دستورالعمل‌هایی را ارزیابی می‌کند و معیارهای ارزیابی آن کدامند؟<br>

✍️ <b>پاسخ تشریحی:</b>

بنچمارک IFEval برای این طراحی شده که بسنجد مدل تا چه حد می‌تواند به دستورالعمل‌هایی که به‌صورت دقیق و قابل‌بررسی نوشته شده‌اند عمل کند. یعنی به‌جای سؤال‌های کلی، به مدل پرامپت‌هایی می‌دهد که داخلشان یک یا چند قید مشخص وجود دارد؛ مثلا «حداقل ۴۰۰ کلمه بنویس»، «در انتها حتماً با کلمه SOUL تمام کن»، «فقط با حروف بزرگ بنویس»، «خروجی را در قالب JSON معتبر بده» و چیزهایی از این جنس که می‌شود بعداً به‌طور خودکار چک کرد رعایت شده‌اند یا نه. برای ارزیابی، معمولاً دو نوع دقت گزارش می‌شود: یکی در سطح خود هر دستور (این‌که برای هر constraint جداگانه مدل درست انجام داده یا نه) و یکی در سطح هر پرامپت؛ یعنی اگر پرامپت چند قید داشته باشد، فقط وقتی نمره کامل می‌گیرد که مدل همه‌ی آن قیدها را به‌درستی رعایت کرده باشد. به این شکل IFEval به‌صورت کمی نشان می‌دهد مدل چقدر واقعاً دستور محور و «حرف‌گوش‌کن» است، نه فقط این‌که جواب ظاهراً مرتبط تولید کند.




<div dir="rtl" style="width: 98%; text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">

#### **ابزار ارزیابی: lm-evaluation-harness**

برای اجرای ارزیابی از **[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)** استفاده می‌کنیم. این فریم‌ورک:

- بیش از ۶۰ بنچمارک استاندارد را پشتیبانی می‌کند.
- از [vLLM](https://github.com/vllm-project/vllm) برای افزایش سرعت استنتاج و مصرف بهینه حافظه پشتیبانی می‌کند.
- خروجی استاندارد و قابل مقایسه تولید می‌کند.

پیشنهاد میشود از vLLM برای افزایش سرعت ارزیابی استفاده کنید.
</div>

In [19]:
%%bash
python -m lm_eval \
  --model hf \
  --model_args pretrained=TinyLlama/TinyLlama-1.1B-Chat-v1.0,dtype=float16 \
  --tasks ifeval \
  --device cuda:0 \
  --batch_size 2 \
  --limit 50 \
  --output_path results_ifeval_base.json \
  --log_samples


hf (pretrained=TinyLlama/TinyLlama-1.1B-Chat-v1.0,dtype=float16), gen_kwargs: (None), limit: 50.0, num_fewshot: None, batch_size: 2
|Tasks |Version|Filter|n-shot|        Metric         |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|-----:|---|------|
|ifeval|      2|none  |     0|prompt_level_strict_acc|0.0600|±  |0.0339|
|      |       |none  |     0|inst_level_strict_acc  |0.0921|±  |N/A   |
|      |       |none  |     0|prompt_level_loose_acc |0.0600|±  |0.0339|
|      |       |none  |     0|inst_level_loose_acc   |0.0921|±  |N/A   |



2025-12-15 14:08:17.704690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765807697.971837   18374 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765807698.045560   18374 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765807698.603422   18374 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765807698.603468   18374 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765807698.603472   18374 computation_placer.cc:177] computation placer alr

<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>

برای ارزیابی کمی توانایی instruction-following، مدل پایه TinyLlama-1.1B-Chat-v1.0 را با استفاده از lm-evaluation-harness روی بنچمارک IFEval ارزیابی کردم (به‌دلیل محدودیت منابع با فلگ `--limit 50` فقط روی ۵۰ نمونه از تسک اجرا شد).
نتایج نشان می‌دهد که دقت strict در سطح پرامپت حدود ۰٫۰۶ است، یعنی فقط تقریباً ۶٪ از پرامپت‌ها تمام دستورالعمل‌های خود را به‌طور کامل رعایت می‌کنند. دقت strict در سطح دستور (inst_level) نیز حدود ۰٫۰۹ است؛ بنابراین مدل تنها حدود ۹٪ از دستورهای ریز را به‌درستی اجرا می‌کند. این اعداد بیان‌گر آن است که TinyLlama به‌صورت پایه، در رعایت دقیق قیدهای پیچیده (مانند تعداد کلمات مشخص، حروف بزرگ، فرمت JSON و …) چندان قوی نیست و اغلب فقط بخشی از دستور را رعایت می‌کند. علاوه بر این، در لاگ ابزار هشدار داده می‌شود که IFEval به‌طور خاص برای مدل‌های چت‌محور طراحی شده و doc_to_text در تنظیمات فعلی لزوماً با chat template دقیق مدل منطبق نیست؛ در نتیجه این نتایج را باید به‌عنوان تخمینی تقریبی در نظر گرفت، نه یک ارزیابی نهایی در مقیاس بزرگ. با این وجود، همین آمار برای مقایسه‌ی رفتار مدل پایه و مدل فاین‌تیون‌شده در مراحل بعدی مفید خواهد بود.


In [20]:
%%bash
python -m lm_eval \
  --model hf \
  --model_args pretrained=./tinyllama_lora_merged,dtype=float16 \
  --tasks ifeval \
  --device cuda:0 \
  --batch_size 2 \
  --limit 50 \
  --output_path results_ifeval_ft.json \
  --log_samples


hf (pretrained=./tinyllama_lora_merged,dtype=float16), gen_kwargs: (None), limit: 50.0, num_fewshot: None, batch_size: 2
|Tasks |Version|Filter|n-shot|        Metric         |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|-----:|---|------|
|ifeval|      2|none  |     0|prompt_level_strict_acc|0.1000|±  |0.0429|
|      |       |none  |     0|inst_level_strict_acc  |0.2632|±  |N/A   |
|      |       |none  |     0|prompt_level_loose_acc |0.1000|±  |0.0429|
|      |       |none  |     0|inst_level_loose_acc   |0.2895|±  |N/A   |



2025-12-15 14:16:07.328389: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765808167.689285   20304 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765808167.778582   20304 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765808168.459536   20304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765808168.459657   20304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765808168.459682   20304 computation_placer.cc:177] computation placer alr

<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>

✍️ **تحلیل نتایج IFEval برای مدل فاین‌تیون‌شده و مقایسه با مدل پایه:**

برای مدل فاین‌تیون‌شده (tinyllama_lora_merged) روی ۵۰ نمونه‌ی IFEval، مقادیر زیر به‌دست آمد:

* `prompt_level_strict_acc = 0.10` → یعنی مدل روی حدود **۱۰٪ پرامپت‌ها** تمام دستورالعمل‌های مربوط به آن پرامپت را به‌طور کامل رعایت کرده است.
* `inst_level_strict_acc = 0.2632` → یعنی در حدود **۲۶٪ از تک‌تک دستورها** (constraints ریز داخل همه‌ی پرامپت‌ها) به‌درستی رعایت شده‌اند.
* `prompt_level_loose_acc = 0.10` → در این زیرمجموعه، عدد loose در سطح پرامپت با strict برابر شده است.
* `inst_level_loose_acc = 0.2895` → در تعریف loose، حدود **۲۹٪** از دستورها (کمی با معیار نرم‌تر) رعایت شده‌اند.

در مقایسه با مدل پایه، که نتایجش این بود:

* `prompt_level_strict_acc = 0.06`
* `inst_level_strict_acc ≈ 0.0921`
* `prompt_level_loose_acc = 0.06`
* `inst_level_loose_acc ≈ 0.0921`

به‌طور عددی می‌توان گفت:

* دقت strict در سطح پرامپت از حدود **۶٪ به ۱۰٪** رسیده است؛ یعنی شانس این‌که مدل کل دستورهای یک پرامپت را کامل رعایت کند تقریباً **۱٫۶ تا ۱٫۷ برابر** بهتر شده، هرچند هنوز پایین است و نشان می‌دهد که اجرای کاملاً دقیق تمام قیدها همچنان کار سختی برای این مدل کوچک است.
* دقت strict در سطح دستور (`inst_level_strict`) از حدود **۹٪ به حدود ۲۶٪** رسیده؛ یعنی تقریباً **۲٫۸ برابر** بهبود. این نشان می‌دهد که مدل فاین‌تیون‌شده، در سطح ریزتر (مثلاً رعایت حروف بزرگ، تعداد جملات، وجود یا عدم وجود کلمات خاص و…) خیلی بهتر از مدل پایه عمل می‌کند و تعداد زیادی از constraintها را درست اجرا می‌کند، حتی اگر همه‌ی آن‌ها را در یک پرامپت به‌طور هم‌زمان نتواند رعایت کند.
* دقت loose در سطح دستور (`inst_level_loose_acc ≈ 0.2895`) هم نسبت به مدل پایه (حدود ۰٫۰۹۲۱) تقریباً **بیش از ۳ برابر** شده است؛ این نشان می‌دهد که اگر معیار ارزیابی را کمی نرم‌تر بگیریم، مدل فاین‌تیون‌شده در اجرای «نسبتاً درست» یا «تقریباً مطابق انتظار» دستورها به شکل محسوسی بهتر از مدل پایه است.

از نظر تفسیری، می‌توان گفت که **Supervised Fine-Tuning با LoRA روی داده‌های instruction-following واقعاً رفتار مدل را در جهت رعایت دستورالعمل‌ها تقویت کرده است**: مدل فاین‌تیون‌شده هم در سطح پرامپت و هم به‌خصوص در سطح دستور، موفقیت بیشتری در اجرای constraintها دارد. این با مشاهده‌ی کیفی تو هم هم‌خوان بود؛ جایی که مدل فاین‌تیون‌شده بیشتر سعی می‌کرد ALL CAPS، ساختار JSON و… را رعایت کند، در حالی که مدل پایه بعضی از این قیدها را کاملاً نادیده می‌گرفت.

با این حال، چند نکته‌ی مهم هم باید در تحلیل ذکر شود. اول این‌که هر دو ارزیابی با فلگ `--limit 50` انجام شده‌اند، یعنی فقط روی ۵۰ نمونه‌ی IFEval؛ بنابراین این اعداد تخمینی هستند و نوسان بالایی دارند (همان‌طور که مقدار Stderr نسبتاً بزرگ است). دوم، خود lm-eval هشدار می‌دهد که تسک IFEval برای مدل‌های چتی خاص (مثل OpenAI/Anthropic) طراحی شده و در تنظیمات فعلی ممکن است promptها دقیقاً با chat template بومی TinyLlama منطبق نباشند؛ پس این نتایج را باید بیشتر به‌عنوان یک **مقایسه‌ی نسبی بین مدل پایه و مدل فاین‌تیون‌شده** در نظر گرفت، نه یک عدد مطلق و نهایی.

در جمع‌بندی می‌توان گفت: بعد از SFT، TinyLlama در بنچمارک IFEval از نظر توانایی instruction-following به‌طور قابل توجهی بهتر شده است؛ مدل فاین‌تیون‌شده هم در احتمال موفقیت کامل روی یک پرامپت و هم در درصد دستورهای ریزِ اجراشده، چند برابر بهتر از مدل پایه عمل می‌کند، هرچند در سطح absolute هنوز فاصله‌ی زیادی با مدل‌های بزرگ و قوی‌تر دارد. این با رفتار کیفی مشاهده‌شده (رعایت بهتر فرمت، حروف بزرگ، JSON و… در کنار تمایل به توضیح‌دادن) هم سازگار است.


<p dir='rtl' style="line-height: 2.0; text-align: right; font-family: Vazir; font-size: 16px; margin-top: 20px; color: white; background-color:rgb(0, 40, 30); padding: 30px; border-radius: 8px;">
🎯 <b>خروجی مورد انتظار:</b><br>
نصب ابزار (clone & pip install) لازم نیست در نوت‌بوک گزارش شود.<br>
اما اسکریپت اجرای ارزیابی + لاگ های خروجی آن باید داخل نوت‌بوک قرار داشته باشند.<br>
در انتهای بخش ارزیابی، یک جدول مقایسه بین مدل پایه و مدل فاین‌تیون‌شده ارائه دهید.
</p>

<div dir='rtl' style='width: 98%; background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
<table style="width:100%; border-collapse:collapse; text-align:center; font-size:14px;">
  <thead>
    <tr style="background:#ffe8a1;">
      <th style="border:1px solid #f0ad4e; padding:6px;">معیار</th>
      <th style="border:1px solid #f0ad4e; padding:6px;">مدل پایه<br>TinyLlama Base</th>
      <th style="border:1px solid #f0ad4e; padding:6px;">مدل فاین‌تیون‌شده<br>TinyLlama LoRA SFT</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border:1px solid #f0ad4e; padding:6px;">prompt_level_strict_acc</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.0600</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.1000</td>
    </tr>
    <tr>
      <td style="border:1px solid #f0ad4e; padding:6px;">inst_level_strict_acc</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.0921</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.2632</td>
    </tr>
    <tr>
      <td style="border:1px solid #f0ad4e; padding:6px;">prompt_level_loose_acc</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.0600</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.1000</td>
    </tr>
    <tr>
      <td style="border:1px solid #f0ad4e; padding:6px;">inst_level_loose_acc</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.0921</td>
      <td style="border:1px solid #f0ad4e; padding:6px;">0.2895</td>
    </tr>
  </tbody>
</table>

<p style="margin-top:10px; line-height:1.8;">
همان‌طور که در جدول دیده می‌شود، پس از انجام SFT با LoRA، دقت مدل در رعایت دستورالعمل‌ها روی بنچمارک IFEval در همه‌ی معیارها بهتر شده است؛
به‌خصوص در سطح دستور (inst_level) که از حدود ۹٪ به حدود ۲۶–۲۹٪ رسیده است.  
توجه شود که هر دو ارزیابی با فلگ <code>--limit 50</code> فقط روی ۵۰ نمونه انجام شده‌اند، بنابراین این اعداد تقریبی‌اند و بیشتر برای مقایسه‌ی نسبی بین مدل پایه و مدل فاین‌تیون‌شده استفاده می‌شوند، نه به‌عنوان ارزیابی نهایی در مقیاس بزرگ.
</p>
</div>